In [66]:
import geopandas as gpd
import pandas as pd 
import numpy as np
from pathlib import Path
from shapely import wkt

In [78]:
rc_lookups = pd.read_csv("output/rc_lookups.csv")
rc_ldc =pd.read_csv("output/retail_centers_with_ldc_retailN.csv")

In [79]:
rc = rc_ldc.merge(
     rc_lookups.drop(columns=['geometry']),
    on='RC_ID',
    how='left'
)
rc

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,geometry,LDC_retailN,LAD24NM,CountyName,RegionName,gsscode
0,6025,16,5,7.0,6.0,0.0,3.0,['Asian Restaurant' 'Electronics Store' 'Café'...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((440445.7388613202 367018.09139...,2.0,North East Derbyshire,Derbyshire,East Midlands,E63008766
1,1193,131,71,86.0,33.0,4.0,8.0,"['Electronics Store' ""Doctor's Office"" 'Drugst...",centre,North East Derbyshire,0,0,MULTIPOLYGON (((439116.7765939621 363407.88751...,131.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
2,6644,14,11,11.0,2.0,1.0,0.0,['Electronics Store' 'Hardware Store' 'Conveni...,centre,Cornwall,0,0,MULTIPOLYGON (((194309.88097327633 76200.71289...,10.0,Cornwall,Cornwall,South West,E63013978
3,6026,16,44,11.0,3.0,2.0,0.0,['Gym' 'Gastropub' 'Brewery' 'Funeral Home' 'B...,centre,North East Derbyshire,0,0,MULTIPOLYGON (((439008.4176877096 364010.85388...,17.0,North East Derbyshire,Derbyshire,East Midlands,E63008809
4,7407,12,4,6.0,3.0,1.0,2.0,"['Hardware Store' ""Doctor's Office"" 'Indian Re...",centre,Chesterfield,0,0,MULTIPOLYGON (((438294.8993448653 371476.61951...,8.0,Chesterfield,Derbyshire,East Midlands,E63008717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9617,2844,46,16,32.0,13.0,1.0,0.0,['Convenience Store' 'Discount Store' 'Restaur...,retail_park,"Armagh City, Banbridge and Craigavon",0,0,MULTIPOLYGON (((111614.72348488995 512113.4056...,59.0,"Armagh City, Banbridge and Craigavon",Armagh,NI,NI
9618,4025,29,22,17.0,8.0,2.0,2.0,['Drugstore' 'Pet Service' 'Convenience Store'...,retail_park,Braintree,0,0,MULTIPOLYGON (((578163.6762067255 233984.52018...,14.0,Braintree,Essex,East of England,E63011054
9619,2676,50,11,34.0,14.0,2.0,0.0,['Textiles Store' 'Electronics Store' 'Sportin...,retail_park,"Bournemouth, Christchurch and Poole",0,0,MULTIPOLYGON (((401222.3074132583 90489.386812...,46.0,"Bournemouth, Christchurch and Poole",Dorset,South West,E63013860
9620,3757,32,22,26.0,6.0,0.0,0.0,['Sporting Goods Retail' 'Convenience Store' '...,retail_park,Glasgow City,0,0,MULTIPOLYGON (((255356.56308415128 666668.6817...,50.0,Glasgow City,City of glasgow,Scotland,S45001180


In [80]:
rc[rc['gsscode'].isna()]['centre_type'].value_counts()  

centre_type
centre         83
retail_park     8
Name: count, dtype: int64

In [16]:
rc.columns

Index(['RC_ID', 'Total_POI_count', 'H3_cells', 'Retail', 'Dining_and_Drinking',
       'Leisure_Travel', 'Community_Health', 'category_names', 'centre_type',
       'LocalAuthorityNM_1', 'LocalAuthorityNM_2', 'LocalAuthorityNM_3',
       'geometry', 'LDC_retailN', 'LAD24NM', 'CountyName', 'RegionName',
       'gsscode'],
      dtype='object')

# apply_hierarchical_classification

In [111]:
def apply_hierarchical_classification(df: pd.DataFrame, retail_park_rc_ids=None) -> pd.DataFrame:
    df = df.copy()
    retail_park_rc_ids = set() if retail_park_rc_ids is None else set(retail_park_rc_ids)

    # 1. HANDLE NA VALUES PROPERLY
    # Instead of filling with "NA" before ranking, we rank first 
    # and then handle the NAs to match the R script logic.
    rank_specs = {
        "RGN_rank": "RegionName",
        "CTY_rank": "CountyName",
        "LAD_rank": "LAD24NM",
        "BUA_rank": "gsscode",
    }
    
    for rank_col, group_col in rank_specs.items():
        # Only rank where the group name is NOT null
        mask = df[group_col].notna()
        df.loc[mask, rank_col] = df[mask].groupby(group_col)["LDC_retailN"].rank(
            method="first", ascending=False
        )
        # If it was NA, give it a massive rank so it's never Rank 1
        df[rank_col] = df[rank_col].fillna(999999)

    df["tr_retailN"] = df["LDC_retailN"]
    df["Classification"] = pd.Series(pd.NA, index=df.index)

    def assign(label: str, mask: pd.Series) -> None:
        df.loc[df["Classification"].isna() & mask, "Classification"] = label

    # TIER 1: Regional (With Manual Overrides)
    assign("Regional Centre", (df["RGN_rank"] == 1))

    # TIER 2: Major Town
    assign("Major Town Centre", (df["CTY_rank"] == 1) & (df["RGN_rank"] != 1) & (df["LDC_retailN"] >= 350))

    # TIER 3: Town Centre
    assign("Town Centre", 
        ((df["CTY_rank"] == 1) & (df["RGN_rank"] != 1)) | 
        ((df["LAD_rank"] == 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1)) | 
        ((df["LDC_retailN"] >= 250) & (df["LDC_retailN"] < 350)) 
    )

    # TIER 4: Market Town
    assign("Market Town", 
        (df["BUA_rank"] == 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150) & (df["tr_retailN"] < 250)
    )

    # TIER 5: District Centre
    assign("District Centre", 
        (df["BUA_rank"] != 1) & (df["LAD_rank"] != 1) & (df["RGN_rank"] != 1) & (df["CTY_rank"] != 1) & 
        (df["tr_retailN"] >= 150)
    )

    # TIER 6: Local Centre
    assign("Local Centre", df["tr_retailN"] < 150)

    # REFINEMENT: Local -> Small Local
    df.loc[(df["Classification"] == "Local Centre") & (df["tr_retailN"] < 100), "Classification"] = "Small Local Centre"

    # FALLBACKS
    df.loc[df["Classification"].isna() & df["LDC_retailN"].notna(), "Classification"] = "Market Town"
    df["Classification"] = df["Classification"].fillna("Small Local Centre")

    # RETAIL PARK ADJUSTMENT
    # Refined to include District Centres if they are actually Retail Parks
    retail_park_mask = (
        (df["centre_type"] == "retail_park") 
    ) & (df["Classification"].isin(["Small Local Centre"]))
    # retail_park_mask = (
    #     (df["centre_type"] == "retail_park") | 
    #     df["RC_ID"].isin(retail_park_rc_ids)
    # ) & (df["Classification"].isin(["Small Local Centre", "Local Centre", "District Centre"]))
    
    df.loc[retail_park_mask, "Classification"] = "Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] < 50), "Classification"] = "Small Retail Park"
    df.loc[(df["Classification"] == "Retail Park") & (df["tr_retailN"] >= 50), "Classification"] = "Large Retail Park"

    return df.sort_values(["RegionName", "RC_ID"]).reset_index(drop=True)

In [112]:
df= apply_hierarchical_classification(rc)
df['Classification'].value_counts()

Classification
Small Local Centre    7083
Small Retail Park     1079
Town Centre            438
Local Centre           399
Market Town            238
District Centre        236
Large Retail Park       83
Major Town Centre       54
Regional Centre         12
Name: count, dtype: int64

# add manual

In [ ]:
def apply_manual_edits(
    df: pd.DataFrame,
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
    manual_hierarchy_csv: str | None = None,
) -> pd.DataFrame:
    """
    Manual classification edits + optional Manual_Hierarchy upgrades.
    rcid_lookup_csv: path to rcid_lookup_old_to_new.csv
    manual_hierarchy_csv: path to Manual_Hierarchy.csv (optional)
    """
    df = df.copy()

    # --- RC_ID mapping (old -> new) ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = dict(zip(lookup["RC_ID"], lookup["new_rcid"]))

    def map_rcid(old_id: str) -> str:
        return old_to_new.get(old_id, old_id)

    # --- Manual classification edits (old RC_IDs in the R code) ---
    manual_classification_old = {
        "RC_EW_2920": "Market Town",
        "RC_SC_332": "Major Town Centre",
        "RC_EW_876": "Major Town Centre",
        "RC_EW_5564": "Major Town Centre",
        "RC_EW_61": "Major Town Centre",
        "RC_EW_3096": "Major Town Centre",
        "RC_EW_3254": "Major Town Centre",
        "RC_EW_3301": "Major Town Centre",
        "RC_EW_3476": "Major Town Centre",
        "RC_EW_3522": "Regional Centre", # plus bristol
        "RC_EW_2751": "Regional Centre", # plus liverpool one
         "RC_SC_280": "Regional Centre", # plus edinburgh
    }

    manual_classification = {map_rcid(k): v for k, v in manual_classification_old.items()}
    for rcid, cls in manual_classification.items():
        df.loc[df["RC_ID"] == rcid, "Classification"] = cls

    # --- Manual hierarchy list (upgrade Town Centre -> Major Town Centre) ---
    if manual_hierarchy_csv:
        manual_path = Path(manual_hierarchy_csv)
        if manual_path.exists():
            rc_manual = pd.read_csv(manual_path)
            rc_manual = rc_manual.loc[
                rc_manual["H_Classification"] == "Major Town Centre", "RC_ID"
            ].dropna()
            rc_manual = [map_rcid(x) for x in rc_manual.astype(str).unique()]

            mask = (df["RC_ID"].isin(rc_manual)) & (df["Classification"] == "Town Centre")
            df.loc[mask, "Classification"] = "Major Town Centre"

    return df

In [119]:
df1 = apply_manual_edits(df)
df1['Classification'].value_counts()

Classification
Small Local Centre    7083
Small Retail Park     1079
Town Centre            431
Local Centre           398
Market Town            239
District Centre        236
Large Retail Park       83
Major Town Centre       58
Regional Centre         15
Name: count, dtype: int64

# shopping centres

In [ ]:
def apply_shopping_outlet_overrides(
    df: pd.DataFrame,
    rc_uk_path: str = "geo/postworkshopcentres.gpkg",
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
) -> pd.DataFrame:
    """
    Manual outlet locations (spatial intersection + explicit ID overrides).
    rc_uk_path: path to postworkshopcentres.gpkg
    rcid_lookup_csv: path to rcid_lookup_old_to_new.csv
    """
    df = df.copy()

    # --- RC_ID mapping (old -> new) ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = dict(zip(lookup["RC_ID"], lookup["new_rcid"]))

    def map_rcid(old_id: str) -> str:
        return old_to_new.get(old_id, old_id)

    # --- Manual outlet locations ---
    outlets = pd.DataFrame(
        [
            ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
            ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
            ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
            ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
            ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
            ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
            ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
            ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
            ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
            ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
            ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
            ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
            ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
            ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
            ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
            ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
            ("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
            ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
            ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
            ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
            ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
            ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
            ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
            ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
            ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
            ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
            ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
            ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
            ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
            ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
            ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
            ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
            ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
            ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
            ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
            ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
            ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
            ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
            ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
            ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
            ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
            ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
            ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
            ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
            ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
            ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
            ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
        ],
        columns=["SubType", "Name", "Lat", "Lon"],
    )
    outlets["Type"] = "Out of Town Shopping Centres"

    outlets_gdf = gpd.GeoDataFrame(
        outlets,
        geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
        crs="EPSG:4326",
    )

    rc_uk = gpd.read_file(rc_uk_path)
    outlets_gdf = outlets_gdf.to_crs(rc_uk.crs)

    outlets_joined = gpd.sjoin(
        outlets_gdf,
        rc_uk[["RC_ID", "geometry"]],
        how="left",
        predicate="intersects",
    )
    outlets_tag = outlets_joined[["Type", "RC_ID"]].dropna(subset=["RC_ID"])

    manual_outlet_rcids_old = {
        "Ashford Designer Outlet": "RC_EW_1312",
        "Chatham Dockside Outlet Centre": "RC_EW_1529",
        "Doncaster Lakeside Village": "RC_EW_762",
        "Swindon Designer Outlet": "RC_EW_3751",
        "Springfields Spalding Outlet": "RC_EW_955",
    }

    manual_override = outlets_gdf[outlets_gdf["Name"].isin(manual_outlet_rcids_old)].copy()
    manual_override["RC_ID"] = manual_override["Name"].map(manual_outlet_rcids_old).map(map_rcid)
    manual_override = manual_override[["Type", "RC_ID"]].dropna(subset=["RC_ID"])

    outlets_tag = pd.concat([outlets_tag, manual_override], ignore_index=True)

    df = df.merge(outlets_tag, on="RC_ID", how="left", sort=False)
    df.loc[df["Type"].notna(), "Classification"] = df.loc[df["Type"].notna(), "Type"]
    df = df.drop(columns=["Type"])

    return df


In [120]:
df2 = apply_shopping_outlet_overrides(df1)
df2['Classification'].value_counts()

Classification
Small Local Centre              7064
Small Retail Park               1077
Town Centre                      422
Local Centre                     393
Market Town                      235
District Centre                  229
Large Retail Park                 83
Major Town Centre                 57
Out of Town Shopping Centres      49
Regional Centre                   15
Name: count, dtype: int64

In [121]:
def apply_out_of_town_sizing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    out_of_town_mask = df["Classification"].isin(["Out of Town Shopping Centres"])

    df.loc[out_of_town_mask & (df["tr_retailN"] < 50), "Classification"] = "Small Shopping Centre"
    df.loc[out_of_town_mask & (df["tr_retailN"] >= 50), "Classification"] = "Large Shopping Centre"

    return df


In [122]:
df3 = apply_out_of_town_sizing(df2)
df3['Classification'].value_counts()

Classification
Small Local Centre       7064
Small Retail Park        1077
Town Centre               422
Local Centre              393
Market Town               235
District Centre           229
Large Retail Park          83
Major Town Centre          57
Large Shopping Centre      43
Regional Centre            15
Small Shopping Centre       6
Name: count, dtype: int64

In [123]:
df3['Classification'].isna().sum()

np.int64(0)

In [129]:
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df3[cols] = df3[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")

In [130]:
df3["geometry"] = df3["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

# Now create GeoDataFrame with CRS
gdf = gpd.GeoDataFrame(df3, geometry="geometry", crs="EPSG:27700")  # change CRS if needed
hex_map = {
    "Regional Centre": "#882255",
    "Major Town Centre": "#332288",
    "Town Centre": "#117733",
    "Market Town": "#44AA99",
    "District Centre": "#999933",
    "Local Centre": "#DDCC77",
    "Small Local Centre": "#88CCEE",
    "Large Retail Park": "#CC6677",
    "Small Retail Park": "#E0A1AB",
    "Large Out of Town Shopping Centre": "#AA4499",
    "Small Out of Town Shopping Centre": "#CD83C1",
}

gdf["HEX_COL"] = gdf["Classification"].map(hex_map)

# Export (pick your path)
out_path = "output/rc2025_classified.shp"
gdf.to_file(out_path, driver="ESRI Shapefile")

/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_11906/1244087030.py:25: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(out_path, driver="ESRI Shapefile")
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total_POI_count' to 'Total_POI_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Dining_and_Drinking' to 'Dining_and'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Leisure_Travel' to 'Leisure_Tr'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Community_Health' to 'Community_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: Runti

In [132]:
geo = df3
geo["geometry"] = geo["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)

geo_gdf = gpd.GeoDataFrame(geo, geometry="geometry", crs="EPSG:27700")
geo_gdf.to_file("output/rc2025_classified.gpkg", driver="GPKG")


# Join the split togther

In [252]:
rc_geo = geo_gdf
# split info
rc_split = gpd.read_file("geo/postworkshopcentres_withsplit.gpkg")

# normalize new split format:
# - RC_ID_split: original field (e.g. 10_1, 10_2, 11)
# - Parent_RC_ID: parent (before underscore)
# - RC_ID: original child id (after underscore) or own id if unsplit
rc_split.rename(columns={"RC_ID": "RC_ID_split"}, inplace=True)
split_parts = rc_split["RC_ID_split"].astype(str).str.split("_", n=1, expand=True)
rc_split["Parent_RC_ID"] = split_parts[0]
rc_split["RC_ID"] = split_parts[1].fillna(split_parts[0])
rc_split["was_split"] = split_parts[1].notna()

# align dtype to rc_geo RC_ID if needed
if "RC_ID" in rc_geo.columns and rc_geo["RC_ID"].dtype != object:
    rc_split["RC_ID"] = pd.to_numeric(rc_split["RC_ID"], errors="coerce")
    rc_split["Parent_RC_ID"] = pd.to_numeric(rc_split["Parent_RC_ID"], errors="coerce")

rc_split.sort_values("RC_ID", ascending=True).head(11)


,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,LocalAuthorityNM_2,LocalAuthorityNM_3,was_split,Original_POI_count,Parent_RC_ID,geometry
2294,1,3798,467,2188.0,957.0,591.0,62.0,['Latin American Restaurant' 'Tourist Informat...,centre,Westminster,None,None,True,30979,1,"MULTIPOLYGON (((528502.555 180280.192, 528522...."
2295,2,3677,427,1405.0,1698.0,519.0,55.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Westminster,Camden,None,True,30979,1,"MULTIPOLYGON (((529315.21 180544.304, 529310.2..."
2312,3,3078,351,1392.0,1210.0,430.0,46.0,['Tourist Information and Service' 'Electronic...,centre,Westminster,Camden,None,True,30979,1,"MULTIPOLYGON (((529954.178 180354.802, 529933...."
2310,4,2517,389,1073.0,1011.0,395.0,38.0,['Electronics Store' 'Gastropub' 'Souvlaki Sho...,centre,Hackney,Tower Hamlets,None,True,30979,1,"MULTIPOLYGON (((532905.685 182028.4, 532880.34..."
2306,5,2480,402,1065.0,1098.0,251.0,66.0,['Electronics Store' 'Gastropub' 'Tourist Info...,centre,Tower Hamlets,City of London,None,True,30979,1,"MULTIPOLYGON (((533086.931 181413.386, 533061...."
2313,6,1989,299,966.0,517.0,138.0,368.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Westminster,None,None,True,30979,1,"MULTIPOLYGON (((527839.367 180988.039, 527819...."
2311,7,1827,382,794.0,798.0,156.0,79.0,"['Electronics Store' ""Doctor's Office"" 'Gastro...",centre,City of London,None,None,True,30979,1,"MULTIPOLYGON (((532193.397 181084.989, 532188...."
4922,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,None,None,False,1496,23,"MULTIPOLYGON (((318162.615 175928.74, 318141.6..."
2152,9,1487,330,704.0,616.0,123.0,44.0,['Tourist Information and Service' 'Gastropub'...,centre,Westminster,None,None,False,1487,24,"MULTIPOLYGON (((528777.754 178632.782, 528772...."
2347,10,1411,417,672.0,509.0,189.0,41.0,"['Electronics Store' ""Doctor's Office"" 'Latin ...",centre,Hackney,Tower Hamlets,None,False,1411,10,"MULTIPOLYGON (((534098.639 182499.135, 534093...."


In [168]:
rc_geo['RC_ID'].unique()

array([  13,   39,   78, ..., 9600, 9603, 9617], shape=(9622,))

In [258]:
rc_work = rc_geo.copy()

# --- split info table (must include RC_ID, was_split, parent_rc_id) ---
# e.g. split_info = gpd.read_file("geo/postworkshopcentres_withsplitinfo.gpkg")[["RC_ID","was_split","parent_rc_id"]]
split_info = rc_split[["RC_ID", "was_split", "Parent_RC_ID"]]

# # 1) attach split info
# rc_work = rc_work.merge(split_info, on="RC_ID", how="left")

# # 2) create group_id (parent if split, else own RC_ID)
# rc_work["group_id"] = rc_work["RC_ID"]
# rc_work.loc[rc_work["was_split"] == True, "group_id"] = rc_work.loc[rc_work["was_split"] == True, "Parent_RC_ID"]

# # 3) aggregate metrics to group level
# group_df = (
#     rc_work.groupby("group_id", dropna=False)
#            .agg(
#                LDC_retailN=("tr_retailN", "sum"),
#                RegionName=("RegionName", "first"),
#                CountyName=("CountyName", "first"),
#                LAD24NM=("LAD24NM", "first"),
#                gsscode=("gsscode", "first"),
#               centre_type=("centre_type", "first")
#            )
#            .reset_index()
#            .rename(columns={"group_id": "RC_ID"})
# )

# # 4) inspect group_df
# group_df.shape


In [256]:
rc_work.sort_values("RC_ID", ascending=True).head(11)


,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification,was_split,Parent_RC_ID,group_id
1484,1,3798,467,2188.0,957.0,591.0,62.0,['Latin American Restaurant' 'Tourist Informat...,centre,Westminster,...,E63012036,2,2,2,2,1933,District Centre,True,1,1
1485,2,3677,427,1405.0,1698.0,519.0,55.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Westminster,...,E63012036,1,1,1,1,1940,Regional Centre,True,1,1
1486,3,3078,351,1392.0,1210.0,430.0,46.0,['Tourist Information and Service' 'Electronic...,centre,Westminster,...,E63012036,3,3,3,3,1456,District Centre,True,1,1
1487,4,2517,389,1073.0,1011.0,395.0,38.0,['Electronics Store' 'Gastropub' 'Souvlaki Sho...,centre,Hackney,...,E63011969,8,7,1,1,988,Town Centre,True,1,1
1488,5,2480,402,1065.0,1098.0,251.0,66.0,['Electronics Store' 'Gastropub' 'Tourist Info...,centre,Tower Hamlets,...,E63012023,4,4,1,1,1355,Town Centre,True,1,1
1489,6,1989,299,966.0,517.0,138.0,368.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Westminster,...,E63012036,11,10,5,5,897,District Centre,True,1,1
1490,7,1827,382,794.0,798.0,156.0,79.0,"['Electronics Store' ""Doctor's Office"" 'Gastro...",centre,City of London,...,E63012025,7,1,1,1,1060,Major Town Centre,True,1,1
7501,8,1496,213,754.0,581.0,138.0,23.0,['Electronics Store' 'Tourist Information and ...,centre,Cardiff,...,W45001208,1,1,1,1,1001,Regional Centre,False,23,8
1491,9,1487,330,704.0,616.0,123.0,44.0,['Tourist Information and Service' 'Gastropub'...,centre,Westminster,...,E63012036,10,9,4,4,900,District Centre,False,24,9
1492,10,1411,417,672.0,509.0,189.0,41.0,"['Electronics Store' ""Doctor's Office"" 'Latin ...",centre,Hackney,...,E63011969,13,12,2,2,848,Large Shopping Centre,False,10,10


In [257]:
group_df.sort_values("RC_ID", ascending=True).head(11)

,RC_ID,LDC_retailN,RegionName,CountyName,LAD24NM,gsscode,centre_type
0,1,14364,London,Greater london,Westminster,E63012036,centre
1,2,2678,Scotland,City of edinburgh,City of Edinburgh,S45001125,centre
2,3,2373,South East,East sussex,Brighton and Hove,E63013666,centre
3,4,2228,North West,Greater manchester,Manchester,E63008401,centre
4,5,3009,Scotland,City of glasgow,Glasgow City,S45001180,centre
5,6,1708,Yorkshire and The Humber,West yorkshire,Leeds,E63007883,centre
6,7,1816,North West,Merseyside,Liverpool,E63008477,centre
7,8,2480,London,Greater london,Islington,E63011980,centre
8,9,2336,London,Greater london,Westminster,E63012036,centre
9,10,2491,East Midlands,Nottinghamshire,Nottingham,E63009088,centre


In [186]:
group_df['RegionName'].value_counts()

RegionName
London                      1343
South East                  1213
North West                  1197
East of England              856
Yorkshire and The Humber     839
West Midlands                803
Scotland                     787
South West                   739
East Midlands                612
Wales                        458
North East                   399
NI                           237
Name: count, dtype: int64

In [179]:
group_df1 = apply_hierarchical_classification(group_df)
group_df1['Classification'].value_counts()

Classification
Small Local Centre    7039
Small Retail Park     1079
Town Centre            415
Local Centre           386
Market Town            237
District Centre        180
Large Retail Park       83
Major Town Centre       52
Regional Centre         12
Name: count, dtype: int64

In [242]:
group_df1

,RC_ID,LDC_retailN,RegionName,CountyName,LAD24NM,gsscode,centre_type,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,Classification
0,10,2491,East Midlands,Nottinghamshire,Nottingham,E63009088,centre,1.0,1.0,1.0,1.0,2491,Regional Centre
1,13,2381,East Midlands,Derbyshire,Derby,E63009183,centre,2.0,1.0,1.0,1.0,2381,Major Town Centre
2,18,1479,East Midlands,Leicestershire,Leicester,E63009743,centre,3.0,1.0,1.0,1.0,1479,Major Town Centre
3,39,923,East Midlands,Northamptonshire,West Northamptonshire,E63010463,centre,4.0,1.0,1.0,1.0,923,Major Town Centre
4,93,586,East Midlands,Lincolnshire,Lincoln,E63008729,centre,6.0,1.0,1.0,1.0,586,Major Town Centre
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9478,9590,4,Yorkshire and The Humber,South yorkshire,Sheffield,E63008489,retail_park,742.0,206.0,101.0,96.0,4,Small Retail Park
9479,9597,3,Yorkshire and The Humber,Lincolnshire,North Lincolnshire,E63008235,retail_park,753.0,104.0,14.0,8.0,3,Small Retail Park
9480,9600,3,Yorkshire and The Humber,South yorkshire,Doncaster,E63008314,retail_park,754.0,211.0,45.0,15.0,3,Small Retail Park
9481,9603,8,Yorkshire and The Humber,West yorkshire,Bradford,E63007920,retail_park,631.0,286.0,66.0,47.0,8,Small Retail Park


In [ ]:
group_df2 = apply_manual_edits(group_df1)
group_df2['Classification'].value_counts()

Classification
Small Local Centre    7039
Small Retail Park     1079
Town Centre            409
Local Centre           385
Market Town            238
District Centre        180
Large Retail Park       83
Major Town Centre       58
Regional Centre         12
Name: count, dtype: int64

In [ ]:
group_df2a = apply_manual_edits2(group_df1)
group_df2a['Classification'].value_counts()

In [188]:
group_df3 = apply_shopping_outlet_overrides(
    group_df2,
    rc_uk_path="geo/postworkshopcentres.gpkg",
    rcid_lookup_csv="output/rcid_lookup_old_to_new.csv",
)
group_df3 = apply_out_of_town_sizing(group_df3)
group_df3["Classification"].value_counts()


Classification
Small Local Centre       7021
Small Retail Park        1077
Town Centre               401
Local Centre              380
Market Town               234
District Centre           172
Large Retail Park          83
Major Town Centre          57
Large Shopping Centre      43
Regional Centre            11
Small Shopping Centre       6
Name: count, dtype: int64

In [191]:
before = group_df2.copy()
after = apply_shopping_outlet_overrides(
    group_df2.copy(),
    rc_uk_path="geo/postworkshopcentres.gpkg",
    rcid_lookup_csv="output/rcid_lookup_old_to_new.csv",
)

comparison = (
    before[["RC_ID", "Classification"]]
    .rename(columns={"Classification": "before"})
    .merge(
        after[["RC_ID", "Classification"]].rename(columns={"Classification": "after"}),
        on="RC_ID",
        how="inner",
    )
)

changed = comparison.loc[comparison["before"] != comparison["after"]]

changed


,RC_ID,before,after
0,10,Regional Centre,Out of Town Shopping Centres
79,1521,Small Local Centre,Out of Town Shopping Centres
105,1968,Local Centre,Out of Town Shopping Centres
106,1968,Local Centre,Out of Town Shopping Centres
632,330,Town Centre,Out of Town Shopping Centres
703,1219,Local Centre,Out of Town Shopping Centres
717,1374,Small Local Centre,Out of Town Shopping Centres
871,3373,Small Local Centre,Out of Town Shopping Centres
1475,17,District Centre,Out of Town Shopping Centres
1505,169,District Centre,Out of Town Shopping Centres


In [ ]:
#5) classify at group level
group_df = apply_hierarchical_classification(group_df)
group_df = apply_manual_edits(group_df)
group_df = apply_shopping_outlet_overrides(group_df)

# 6) join group classification back to parts
rc_work = rc_work.merge(
    group_df[["RC_ID", "Classification"]],
    left_on="group_id", right_on="RC_ID",
    how="left", suffixes=("", "_group")
)

rc_work["Classification_join"] = rc_work["Classification_group"]

# 7) cleanup helper columns (optional)
rc_work = rc_work.drop(columns=["Classification_group", "group_id", "was_split", "parent_rc_id"], errors="ignore")


In [230]:
# wrap up as a function
def classify_using_split_groups(
    df: pd.DataFrame,
    split_df: pd.DataFrame,
    parent_col: str = "Parent_RC_ID",   # in split_df
    split_col: str = "was_split",       # in split_df
    id_col: str = "RC_ID",
) -> pd.DataFrame:
    df = df.copy()

    # attach split info
    df = df.merge(split_df[[id_col, split_col, parent_col]], on=id_col, how="left")

    # group id: parent if split, else own id
    df["group_id"] = df[id_col]
    df.loc[df[split_col] == True, "group_id"] = df.loc[df[split_col] == True, parent_col]

    # look up admin fields from the group_id (parent RC_ID when split)
    admin_cols = ["RegionName", "CountyName", "LAD24NM", "gsscode", "centre_type"]
    admin_lookup = (
        df[[id_col] + admin_cols]
          .drop_duplicates(subset=[id_col])
          .set_index(id_col)
    )

    # aggregate metrics + attach admin fields for the group_id itself
    group_df = (
        df.groupby("group_id", dropna=False)
          .agg(
              LDC_retailN=("tr_retailN", "sum"),
          )
          .reset_index()
          .rename(columns={"group_id": "RC_ID"})
          .merge(admin_lookup, left_on="RC_ID", right_index=True, how="left")
    )

    # classify at group level
    group_df = apply_hierarchical_classification(group_df)
    group_df = apply_manual_edits(group_df)
    group_df = apply_shopping_outlet_overrides(group_df)
    group_df = apply_out_of_town_sizing(group_df)

    # bring group classification back to original rows
    df = df.merge(
        group_df[["RC_ID", "Classification"]],
        left_on="group_id", right_on="RC_ID",
        how="left", suffixes=("", "_group")
    )
    df["Classification_join"] = df["Classification_group"]

    # cleanup
    df = df.drop(columns=["Classification_group", "group_id", split_col, parent_col], errors="ignore")

    return df


In [235]:
df_join = classify_using_split_groups(
    df= rc_geo,
    split_df=rc_split,
    parent_col="Parent_RC_ID",
    split_col="was_split",
)
df_join['Classification_join'].value_counts()

Classification_join
Small Local Centre       7021
Small Retail Park        1077
Town Centre               430
Local Centre              380
Market Town               234
District Centre           184
Major Town Centre          86
Large Retail Park          83
Regional Centre            67
Large Shopping Centre      60
Small Shopping Centre       6
Name: count, dtype: int64

In [236]:
df_join = df_join.rename(
    columns={
        "Classification": "before_classification",
        "Classification_join": "Classification",
    }
)
df_join

,RC_ID,Total_POI_count,H3_cells,Retail,Dining_and_Drinking,Leisure_Travel,Community_Health,category_names,centre_type,LocalAuthorityNM_1,...,RegionName,gsscode,RGN_rank,CTY_rank,LAD_rank,BUA_rank,tr_retailN,before_classification,RC_ID_group,Classification
0,13,1361,337,756.0,454.0,119.0,32.0,['Tourist Information and Service' 'Electronic...,centre,Derby,...,East Midlands,E63009183,1,1,1,1,1138,Regional Centre,13,Major Town Centre
1,39,1011,506,589.0,299.0,94.0,29.0,['Electronics Store' 'Gastropub' 'Tourist Info...,centre,West Northamptonshire,...,East Midlands,E63010463,2,1,1,1,923,Major Town Centre,39,Major Town Centre
2,78,808,139,422.0,276.0,98.0,12.0,"['Electronics Store' ""Doctor's Office"" 'Touris...",centre,Nottingham,...,East Midlands,E63009088,5,1,1,1,612,Major Town Centre,10,Large Shopping Centre
3,82,798,120,497.0,234.0,59.0,8.0,['Tourist Information and Service' 'Gastropub'...,centre,Leicester,...,East Midlands,E63009743,3,1,1,1,672,Major Town Centre,18,Major Town Centre
4,93,765,120,447.0,241.0,60.0,17.0,"['Electronics Store' 'Gastropub' ""Doctor's Off...",centre,Lincoln,...,East Midlands,E63008729,6,1,1,1,586,Major Town Centre,93,Major Town Centre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9623,9590,2,19,2.0,0.0,0.0,0.0,['Furniture and Home Store'],retail_park,Sheffield,...,Yorkshire and The Humber,E63008489,755,210,105,100,4,Small Retail Park,9590,Small Retail Park
9624,9597,1,7,1.0,0.0,0.0,0.0,['Outdoor Supply Store'],retail_park,North Lincolnshire,...,Yorkshire and The Humber,E63008235,765,104,14,8,3,Small Retail Park,9597,Small Retail Park
9625,9600,1,35,1.0,0.0,0.0,0.0,['Hardware Store'],retail_park,Doncaster,...,Yorkshire and The Humber,E63008314,766,213,44,14,3,Small Retail Park,9600,Small Retail Park
9626,9603,1,26,1.0,0.0,0.0,0.0,['Furniture and Home Store'],retail_park,Bradford,...,Yorkshire and The Humber,E63007920,640,288,65,46,8,Small Retail Park,9603,Small Retail Park


In [237]:
# export
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df_join[cols] = df_join[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")
df_join["geometry"] = df_join["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)
out_gdf = gpd.GeoDataFrame(df_join, geometry="geometry", crs="EPSG:27700")
out_path = "output/rc2025_classified_joined.shp"

out_gdf.to_file(out_path, driver="ESRI Shapefile")

/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_11906/2694130544.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  out_gdf.to_file(out_path, driver="ESRI Shapefile")
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total_POI_count' to 'Total_POI_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Dining_and_Drinking' to 'Dining_and'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Leisure_Travel' to 'Leisure_Tr'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Community_Health' to 'Community_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: R

In [253]:
cols = ["RGN_rank", "CTY_rank", "LAD_rank", "BUA_rank", "tr_retailN"]
df_join[cols] = df_join[cols].apply(pd.to_numeric, errors="coerce").round().astype("int64")
df_join["geometry"] = df_join["geometry"].apply(
    lambda geom: wkt.loads(geom) if isinstance(geom, str) else geom
)
out_gdf = gpd.GeoDataFrame(df_join, geometry="geometry", crs="EPSG:27700")
out_path = "output/rc2025_classified_joined.shp"

out_gdf.to_file(out_path, driver="ESRI Shapefile")
out_gdf.to_file("output/rc2025_classified_joined.gpkg", driver="GPKG")

/var/folders/55/fvwvclfd7t161kfvx0y7f0hm0000gn/T/ipykernel_11906/876672368.py:9: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  out_gdf.to_file(out_path, driver="ESRI Shapefile")
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Total_POI_count' to 'Total_POI_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Dining_and_Drinking' to 'Dining_and'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Leisure_Travel' to 'Leisure_Tr'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Community_Health' to 'Community_'
  ogr_write(
/opt/anaconda3/envs/geods/lib/python3.13/site-packages/pyogrio/raw.py:723: Run

# redo the manual edits

In [214]:
def apply_manual_edits2(
    df: pd.DataFrame,
    rcid_lookup_csv: str = "output/rcid_lookup_old_to_new.csv",
    manual_hierarchy_csv: str | None = None,
    id_col: str = "RC_ID",          # column in df to update (new ids)
    group_col: str | None = None,   # optional: group_id column if you have it
) -> pd.DataFrame:
    """
    Manual classification edits + optional Manual_Hierarchy upgrades.
    Handles one-to-many old -> new RC_ID mapping by applying to all new RC_IDs.
    """
    df = df.copy()

    # --- build old -> list of new mapping ---
    lookup = pd.read_csv(rcid_lookup_csv)
    old_to_new = {}

    for old_id, new_id in zip(lookup["RC_ID"], lookup["new_rcid"]):
        old_to_new.setdefault(old_id, []).append(new_id)

    def map_old_to_new_list(old_id: str):
        # if no mapping exists, fall back to original
        return old_to_new.get(old_id, [old_id])

    # --- Manual classification edits (old RC_IDs in the R code) ---
    manual_classification_old = {
        "RC_EW_2920": "Market Town",
        "RC_SC_332": "Major Town Centre",
        "RC_EW_876": "Major Town Centre",
        "RC_EW_5564": "Major Town Centre",
        "RC_EW_61": "Major Town Centre",
        "RC_EW_3096": "Major Town Centre",
        "RC_EW_3254": "Major Town Centre",
        "RC_EW_3301": "Major Town Centre",
        "RC_EW_3476": "Major Town Centre",
        "RC_EW_3522": "Regional Centre",  # plus Bristol
        "RC_EW_2751": "Regional Centre",  # Liverpool One
        "RC_SC_280": "Regional Centre",   # Edinburgh
    }

    for old_id, cls in manual_classification_old.items():
        new_ids = map_old_to_new_list(old_id)
        mask = df[id_col].isin(new_ids)
        if group_col:
            mask = mask | df[group_col].isin(new_ids)
        df.loc[mask, "Classification"] = cls

    # --- Manual hierarchy list (upgrade Town Centre -> Major Town Centre) ---
    if manual_hierarchy_csv:
        manual_path = Path(manual_hierarchy_csv)
        if manual_path.exists():
            rc_manual = pd.read_csv(manual_path)
            rc_manual = rc_manual.loc[
                rc_manual["H_Classification"] == "Major Town Centre", "RC_ID"
            ].dropna()

            # expand old IDs to all new IDs
            new_ids = []
            for old_id in rc_manual.astype(str).unique():
                new_ids.extend(map_old_to_new_list(old_id))

            mask = df[id_col].isin(new_ids) & (df["Classification"] == "Town Centre")
            if group_col:
                mask = mask | (df[group_col].isin(new_ids) & (df["Classification"] == "Town Centre"))
            df.loc[mask, "Classification"] = "Major Town Centre"

    return df


In [215]:
df_join2 = df = apply_manual_edits2(df_join, "output/rcid_lookup_old_to_new.csv", group_col="group_id")
df_join2['Classification_join'].value_counts()

Classification_join
Small Local Centre       7021
Small Retail Park        1077
Town Centre               430
Local Centre              380
Market Town               234
District Centre           184
Major Town Centre          86
Large Retail Park          83
Regional Centre            67
Large Shopping Centre      60
Small Shopping Centre       6
Name: count, dtype: int64

# check for the outlets

In [ ]:

# build outlets DataFrame 
outlets = pd.DataFrame(
    [
        ("Regional Shopping Centre", "Westfield London", 51.50768597440922, -0.22088352838801395),
        ("Regional Shopping Centre", "Metrocentre", 54.957214904105754, -1.6674536751658506),
        ("Regional Shopping Centre", "Trafford Centre", 53.465100417291744, -2.347637851106119),
        ("Regional Shopping Centre", "Westfield Stratford City", 51.54425552636472, -0.004332458383472109),
        ("Regional Shopping Centre", "Bluewater", 51.43912203176112, 0.27160383738290644),
        ("Regional Shopping Centre", "intu Merry Hill", 52.48138718476229, -2.110658395511001),
        ("Regional Shopping Centre", "Lakeside Shopping Centre", 51.489406902239956, 0.2853764470235825),
        ("Regional Shopping Centre", "Meadowhall", 53.41411230589209, -1.4111151687565566),
        ("Regional Shopping Centre", "East Kilbride Shopping Centre", 55.76021412051323, -4.17708950101207),
        ("Regional Shopping Centre", "Silverburn Centre", 55.82112954772368, -4.341253330960348),
        ("Regional Shopping Centre", "intu Braehead", 55.87600450967236, -4.364642953985305),
        ("Regional Shopping Centre", "White Rose Centre", 53.75868081616114, -1.5734907774777223),
        ("Large Outlet Shopping Centre", "Ashford Designer Outlet", 51.137387301260496, 0.8801687441992894),
        ("Large Outlet Shopping Centre", "Astle Park West Bromwich", 52.520103339621166, -1.9960259925402608),
        ("Large Outlet Shopping Centre", "Bicester Outlet Village", 51.89237714202758, -1.1560770883204268),
        ("Large Outlet Shopping Centre", "Bideford Atlantic Village", 51.00755545946937, -4.237327761297502),
        ("Large Outlet Shopping Centre", "Burberry Factory Outlet London", 51.54651001026377, -0.05093197052890261),
        ("Large Outlet Shopping Centre", "Castleford Junction 32 Outlet", 53.712463797168354, -1.3388364244282458),
        ("Large Outlet Shopping Centre", "Chatham Dockside Outlet Centre", 51.40160304040388, 0.5377104489852201),
        ("Large Outlet Shopping Centre", "Cheshire Oaks Designer Outlet", 53.26539272808919, -2.8809029974792315),
        ("Large Outlet Shopping Centre", "Clacton Factory Outlet", 51.81858769266023, 1.168209463308007),
        ("Large Outlet Shopping Centre", "Clarks Village Outlet Somerset", 51.12833201493458, -2.741588552823589),
        ("Large Outlet Shopping Centre", "Dalton Park Outlet", 54.81537131857434, -1.3739226162513756),
        ("Large Outlet Shopping Centre", "Doncaster Lakeside Village", 53.50769203205623, -1.1135984485990036),
        ("Large Outlet Shopping Centre", "Dover De Bradelei Wharf Outlet", 51.12211019226935, 1.3141697547138849),
        ("Large Outlet Shopping Centre", "East Midlands Outlet Junction 28", 53.108083884731556, -1.3126583496100666),
        ("Large Outlet Shopping Centre", "Freeport Braintree", 51.870070700810075, 0.5714801267433955),
        ("Large Outlet Shopping Centre", "Freeport Fleetwood Outlet", 53.91786152727727, -3.0118505404061744),
        ("Large Outlet Shopping Centre", "Freeport Talke Stoke-on-Trent", 53.070597467076176, -2.263078253210015),
        ("Large Outlet Shopping Centre", "Gloucester Quays Outlet", 51.85953205593224, -2.2516156614946916),
        ("Large Outlet Shopping Centre", "Hertfordshire Galleria Outlet Shopping", 51.76141229905345, -0.24021860677316503),
        ("Large Outlet Shopping Centre", "Hornsea Freeport Outlet", 53.898485374967684, -0.1724229262407349),
        ("Large Outlet Shopping Centre", "Kilver Court Designer Village", 51.19063584650441, -2.5353243560244905),
        ("Large Outlet Shopping Centre", "London Designer Outlet Wembley", 51.55646778421662, -0.28320440479433673),
        ("Large Outlet Shopping Centre", "Lowry Outlet Salford Quays", 53.47092253822526, -2.292514874626797),
        ("Large Outlet Shopping Centre", "Newcastle Royal Quays Outlet", 54.99532901611218, -1.4642812945442476),
        ("Large Outlet Shopping Centre", "Portsmouth Gunwharf Quays", 50.79546395254288, -1.103824062300293),
        ("Large Outlet Shopping Centre", "Springfields Spalding Outlet", 52.799438891276665, -0.126329033767297),
        ("Large Outlet Shopping Centre", "Swindon Designer Outlet", 51.56272470333755, -1.7971899666540831),
        ("Large Outlet Shopping Centre", "The Mill Outlet Batley", 53.7129199988173, -1.6251337290018772),
        ("Large Outlet Shopping Centre", "York Designer Outlet", 53.92233415376806, -1.0768705880781113),
        ("Large Outlet Shopping Centre", "Banbridge Village Outlet", 54.33172185692118, -6.276147653115943),
        ("Large Outlet Shopping Centre", "Cribbs Causeway", 51.524552488774034, -2.5955220725284307),
        ("Large Outlet Shopping Centre", "Bridgend Designer Outlets Wales", 51.532410963682594, -3.576839020076668),
        ("Large Outlet Shopping Centre", "Gretna Gateway Outlet Village", 54.99688074646182, -3.056523264134829),
        ("Large Outlet Shopping Centre", "Livingston Designer Outlet", 55.88310933797727, -3.520137944032474),
        ("Large Outlet Shopping Centre", "Sterling Mills Outlet", 56.14927287877639, -3.740053433726151),
    ],
    columns=["SubType", "Name", "Lat", "Lon"],
)
outlets["Type"] = "Out of Town Shopping Centres"

# Make GeoDataFrame
outlets_gdf = gpd.GeoDataFrame(
    outlets,
    geometry=gpd.points_from_xy(outlets["Lon"], outlets["Lat"]),
    crs="EPSG:4326",
)

# Export
outlets_gdf.to_file("output/outlets.shp", driver="ESRI Shapefile")
